In [ ]:
# === SETUP ===
import os, sys, yaml, json, subprocess, shutil, urllib.request, zipfile, glob
os.chdir('/kaggle/working')

# Copy kit files from Dataset to working
kit_srcs = glob.glob('/kaggle/input/*')
kit_src = next((s for s in kit_srcs if os.path.isdir(os.path.join(s, 'src'))), None)
if kit_src:
    for item in os.listdir(kit_src):
        if item.startswith('.'): continue
        s = os.path.join(kit_src, item)
        d = os.path.join('/kaggle/working', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
print('Kit root:', os.getcwd())


In [ ]:
# === DOWNLOAD WAN2.2-S2V Q8 GGUF ===
cfg = yaml.safe_load(open('config.yaml'))
models_dir = cfg['paths']['models']
os.makedirs(models_dir, exist_ok=True)

model_url = 'https://huggingface.co/Kijai/Wan2.1-GGUF/resolve/main/wan2.1-s2v-Q8_0.gguf'
model_path = os.path.join(models_dir, 'wan2.1-s2v-Q8_0.gguf')

if not os.path.exists(model_path):
    print('Downloading Wan2.2-S2V Q8 GGUF (~14GB)...')
    urllib.request.urlretrieve(model_url, model_path)
    print(f'Done: {os.path.getsize(model_path)/1e9:.1f} GB')
else:
    print(f'Cached: {os.path.getsize(model_path)/1e9:.1f} GB')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config.yaml'
cfg['active_profile'] = 'kaggle_2xt4'
yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)
print('Profile: kaggle_2xt4 (FREE, 2x T4)')


In [ ]:
# STEP 1: set Kaggle-safe config (2xT4, 768x432, 30 steps, free)
import os, yaml
os.chdir('/kaggle/working/film_kit_v3')
cfg = yaml.safe_load(open('config_confrontation.yaml'))
cfg['active_profile'] = 'kaggle_2xt4'
cfg['rate_per_gpu_hr'] = 0.0
cfg['width'] = 768
cfg['height'] = 432
cfg['steps'] = 30
cfg['upscale'] = 2.0
cfg['rejection_samples'] = 3
cfg['max_clip_seconds'] = 8
# point models at Kaggle mount
cfg.setdefault('paths', {})['models'] = '/kaggle/input/moment-models'
yaml.dump(cfg, open('config_confrontation.yaml','w'), default_flow_style=False, sort_keys=False)
print('config set: kaggle_2xt4, 768x432, 30 steps, 2x upscale, rate=$0')
print('final output res after upscale:', 768*2, 'x', 432*2)


In [ ]:
# STEP 2: install + launch ComfyUI headless (same as before)
import os, subprocess, time, urllib.request
os.chdir('/kaggle/working/film_kit_v3')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config_confrontation.yaml'

COMFY = '/kaggle/working/ComfyUI'
if not os.path.isdir(COMFY):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI', COMFY], check=True)
    subprocess.run(['pip', '-q', 'install', '-r', f'{COMFY}/requirements.txt'], check=True)

CN = f'{COMFY}/custom_nodes'; os.makedirs(CN, exist_ok=True)
nodes = {
    'ComfyUI-GGUF':             'https://github.com/city96/ComfyUI-GGUF',
    'ComfyUI-WanVideoWrapper':  'https://github.com/kijai/ComfyUI-WanVideoWrapper',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
}
for name, url in nodes.items():
    d = os.path.join(CN, name)
    if not os.path.isdir(d):
        subprocess.run(['git', 'clone', '--depth', '1', url, d])
        req = os.path.join(d, 'requirements.txt')
        if os.path.exists(req):
            subprocess.run(['pip', '-q', 'install', '-r', req])

if os.path.isdir('/kaggle/input/moment-models'):
    open(f'{COMFY}/extra_model_paths.yaml', 'w').write(
        'kaggle:
  base_path: /kaggle/input/moment-models
'
        '  checkpoints: .
  diffusion_models: .
  unet: .
  clip: .
  vae: .
'
        '  clip_vision: .
  loras: .
  upscale_models: .
  controlnet: .
'
        '  instantid: .
  insightface: .
  audio_encoders: .
')

def _up():
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/', timeout=3); return True
    except Exception:
        return False

if not _up():
    log = open('/kaggle/working/comfyui.log', 'w')
    subprocess.Popen(['python3', f'{COMFY}/main.py', '--listen', '127.0.0.1', '--port', '8188',
                      '--cache-none', '--disable-smart-memory'],
                     cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        if _up(): break
        time.sleep(3)
print('ComfyUI reachable:', _up())
if not _up():
    print('Check log: !tail -40 /kaggle/working/comfyui.log')


In [ ]:
# STEP 3: RUN THE STILLS PATH (the actual render)
# This is the one cell that does the work.
# It runs: audit -> preflight -> forge -> cost meter -> render stills -> voice -> score -> sfx -> finale -> stitch
# ~40 min on 2xT4. $0.
import os
os.chdir('/kaggle/working/film_kit_v3')
os.environ['KIT_CONFIG'] = 'config_confrontation.yaml'
os.environ['KIT_RENDER_MODE'] = 'stills'
os.environ['CONFIRM_FULL_RUN'] = '1'
os.environ['RATE_PER_GPU_HR'] = '0'
!bash MASTER.sh --stills-only


In [ ]:
# STEP 4: download the finished film
import os, shutil
os.chdir('/kaggle/working/film_kit_v3')
out = 'output_confrontation/THE_CONFRONTATION_CLIP.mp4'
if os.path.exists(out):
    shutil.copy(out, '/kaggle/working/THE_CONFRONTATION_CLIP.mp4')
    print('Film ready -> /kaggle/working/THE_CONFRONTATION_CLIP.mp4')
    print('Download from the Output tab on the right.')
else:
    print('Film not found at', out)
    print('Check /tmp/confrontation_work/ for intermediate files.')
    print('If render was interrupted, re-run STEP 3 -- finished beats auto-skip.')


In [ ]:
# === QUALITY GATE + PACKAGE ===
!python3 src/qc.py

out_dir, targets = 'output', ['THE_MOMENT.mp4', 'THE_MOMENT_MASTER_4444.mov']
existing = [f for f in targets if os.path.exists(os.path.join(out_dir, f))]
zip_path = '/kaggle/working/THE_MOMENT_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in existing:
        z.write(os.path.join(out_dir, f), f)
print(f'Delivery zip: {zip_path}')
print('Download from Kaggle Output tab.')
